<a href="https://colab.research.google.com/github/sana200420/naari-ai/blob/sana%2Ftest-protection/retrieval/scripts/link_and_baseline_gold_queries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Link gold queries with fused search + produce the dense/sparse/fused ablation table

Supersedes `link_gold_queries.ipynb`. That notebook used dense-only top-1 to link
the 280 `Gold_KB_final_280` questions to a `correct_answer_id`, and found a
52% category-mismatch rate (146/280) -- real signal, not noise, but dense-only
search is the weakest configuration available (no sparse leg, no rerank), so
some of that is exactly what Lever 3 exists to fix.

This notebook:
1. Runs **dense**, **sparse**, and **fused (RRF)** search for all 280 questions
2. Uses **fused's top-1** as the ground-truth `correct_answer_id` (better
   matching accuracy than dense-alone -- still flagged for human review, not
   silently trusted)
3. Measures Recall@1/5/20 for dense-only and sparse-only **against that fused
   ground truth** -- this is the Lever 3 ablation table, and it reuses the
   dense-only linking work as one leg of it rather than throwing it away

**Caveat, stated plainly:** fused's own "recall" against a ground truth defined
*by its own top-1* is close to tautological -- of course fused finds what fused
said was correct. The meaningful numbers here are how often dense-only and
sparse-only *independently* land on the same answer fused converged on. Once a
person has reviewed the flagged rows (category mismatch or low confidence),
rerun the Recall cells against the corrected `correct_answer_id` column for a
cleaner number.

Needs the same Qdrant credentials as before, and **Runtime -> Change runtime
type -> T4 GPU**.

In [1]:
!pip install -q qdrant-client FlagEmbedding

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.5/250.5 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 83.1 MB/s eta 0:00:00


In [2]:
!rm -rf naari-ai
!git clone --branch sana/test-protection --depth 1 https://github.com/sana200420/naari-ai.git
%cd naari-ai

import sys
sys.path.insert(0, ".")

from retrieval.normalize import normalize_sd
from retrieval.search import HybridRetriever, reciprocal_rank_fusion

print("cloned + imported OK")

Cloning into 'naari-ai'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 83 (delta 6), reused 59 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 716.69 KiB | 4.91 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/naari-ai
cloned + imported OK


In [3]:
import csv

with open("eval/gold_kb_final_280_raw.csv", encoding="utf-8-sig", newline="") as f:
    gold_rows = list(csv.DictReader(f))

with open("knowledge_base/Womens_Health_KB - 2000_final.csv", encoding="utf-8", newline="") as f:
    kb_rows = list(csv.DictReader(f))

sindhi_cats_order = []
seen_cats = set()
for r in kb_rows:
    if r["category"] not in seen_cats:
        sindhi_cats_order.append(r["category"]); seen_cats.add(r["category"])

EN_CATEGORY_ORDER = [
    "Menstrual Health & Periods", "Mental Health & Emotional Well-being",
    "Pregnancy & Maternal Health", "PCOS & Hormonal Health",
    "Women's Nutrition & Wellness", "Menopause & Menopausal Health",
    "Fertility & Reproductive Health", "Vaginal & Personal Hygiene",
]
sd_to_en_category = dict(zip(sindhi_cats_order, EN_CATEGORY_ORDER))

print(f"{len(gold_rows)} gold rows, {len(kb_rows)} KB rows, category map built")

280 gold rows, 2000 KB rows, category map built


In [4]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
print("model loaded")

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model loaded


In [5]:
from getpass import getpass
from qdrant_client import QdrantClient

QDRANT_URL = getpass("Qdrant cluster URL: ")
QDRANT_API_KEY = getpass("Qdrant API key: ")

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

def embed_fn(text):
    normalized = normalize_sd(text)
    out = model.encode([normalized], return_dense=True, return_sparse=True, return_colbert_vecs=False)
    return {"dense": out["dense_vecs"][0].tolist(), "sparse": out["lexical_weights"][0]}

retriever = HybridRetriever(client, embed_fn=embed_fn)
print("retriever ready, collection points:", client.get_collection("naari_ai_kb").points_count)

Qdrant cluster URL: ··········
Qdrant API key: ··········
retriever ready, collection points: 3999


In [6]:
TOP_K = 20
linked = []

for i, row in enumerate(gold_rows, start=1):
    query = row["Question"]

    dense_rows = retriever.dense_search(query, top_k=TOP_K)
    sparse_rows = retriever.sparse_search(query, top_k=TOP_K)

    dense_ids = [r["answer_id"] for r in dense_rows]
    sparse_ids = [r["answer_id"] for r in sparse_rows]
    fused = reciprocal_rank_fusion([dense_ids, sparse_ids])
    fused_ids = [aid for aid, _score in fused]

    row_by_id = {r["answer_id"]: r for r in dense_rows + sparse_rows}
    top1_id = fused_ids[0] if fused_ids else None
    top1_row = row_by_id.get(top1_id)
    top1_cat_en = sd_to_en_category.get(top1_row["category"], top1_row["category"]) if top1_row else ""

    linked.append({
        "query_id": f"gold_{row['ID']}",
        "query": query,
        "stated_category": row["Category"],
        "stated_subcategory": row["Subcategory"],
        "correct_answer_id": top1_id,
        "fused_top1_category_en": top1_cat_en,
        "category_match": (top1_cat_en == row["Category"]),
        "dense_top1_id": dense_ids[0] if dense_ids else None,
        "dense_top1_score": f"{dense_rows[0]['score']:.4f}" if dense_rows else "",
        "sparse_top1_id": sparse_ids[0] if sparse_ids else None,
        "sparse_top1_score": f"{sparse_rows[0]['score']:.4f}" if sparse_rows else "",
        "low_confidence": (dense_rows[0]["score"] < 0.6) if dense_rows else True,
        "gold_own_answer": row["Answer"],
        "gold_own_source": row["Source"],
        "_dense_ids": dense_ids,
        "_sparse_ids": sparse_ids,
        "_fused_ids": fused_ids,
    })

    if i % 40 == 0:
        print(f"linked {i}/{len(gold_rows)}")

print(f"done, {len(linked)} rows linked")

linked 40/280
linked 80/280
linked 120/280
linked 160/280
linked 200/280
linked 240/280
linked 280/280
done, 280 rows linked


In [7]:
import csv

OUT_PATH = "eval/gold_eval_280_linked.csv"
public_fields = [k for k in linked[0].keys() if not k.startswith("_")]
with open(OUT_PATH, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=public_fields)
    w.writeheader()
    for r in linked:
        w.writerow({k: r[k] for k in public_fields})

mismatches = [r for r in linked if not r["category_match"]]
low_conf = [r for r in linked if r["low_confidence"]]
print(f"wrote {OUT_PATH}")
print(f"category mismatches (fused top-1 vs stated category): {len(mismatches)}/{len(linked)}")
print(f"low-confidence dense matches (<0.6): {len(low_conf)}/{len(linked)}")
print("-> review these before treating correct_answer_id as final ground truth.")

wrote eval/gold_eval_280_linked.csv
category mismatches (fused top-1 vs stated category): 129/280
low-confidence dense matches (<0.6): 2/280
-> review these before treating correct_answer_id as final ground truth.


## Ablation table: dense-only vs sparse-only vs fused, Recall@1/5/20

In [8]:
def recall_at_k(ground_truth_key, ranked_key, k):
    hits = 0
    for r in linked:
        gt = r[ground_truth_key]
        ranked = r[ranked_key][:k]
        if gt in ranked:
            hits += 1
    return hits / len(linked)

results = {}
for leg, key in [("dense", "_dense_ids"), ("sparse", "_sparse_ids"), ("fused", "_fused_ids")]:
    results[leg] = {f"recall@{k}": recall_at_k("correct_answer_id", key, k) for k in (1, 5, 20)}

for leg, metrics in results.items():
    print(leg, metrics)

dense {'recall@1': 0.4642857142857143, 'recall@5': 0.8964285714285715, 'recall@20': 1.0}
sparse {'recall@1': 0.5714285714285714, 'recall@5': 0.925, 'recall@20': 1.0}
fused {'recall@1': 1.0, 'recall@5': 1.0, 'recall@20': 1.0}


In [9]:
import datetime

lines = []
lines.append("# Phase 1 dense/sparse/fused ablation -- gold_eval_280 (PROVISIONAL)\n")
lines.append(f"Generated: {datetime.datetime.utcnow().isoformat()}Z\n")
lines.append("\n")
lines.append("**Ground truth = fused search's own top-1 answer**, not independently "
             "human-verified yet. Fused's row is close to tautological (it is being "
             "measured against itself) -- the meaningful comparison is dense-only and "
             "sparse-only against that same reference point. "
             f"{len(mismatches)}/280 rows are flagged for category mismatch and "
             f"{len(low_conf)}/280 for low confidence; re-run after human review for a "
             "trustworthy final number.\n\n")
lines.append("| Leg | Recall@1 | Recall@5 | Recall@20 |\n")
lines.append("|---|---:|---:|---:|\n")
for leg in ("dense", "sparse", "fused"):
    m = results[leg]
    lines.append(f"| {leg} | {m['recall@1']:.3f} | {m['recall@5']:.3f} | {m['recall@20']:.3f} |\n")

with open("eval/results.md", "a", encoding="utf-8") as f:
    f.write("\n\n" + "".join(lines))

print("appended to eval/results.md")
print("".join(lines))

appended to eval/results.md
# Phase 1 dense/sparse/fused ablation -- gold_eval_280 (PROVISIONAL)
Generated: 2026-09-01T08:45:47.046624Z

**Ground truth = fused search's own top-1 answer**, not independently human-verified yet. Fused's row is close to tautological (it is being measured against itself) -- the meaningful comparison is dense-only and sparse-only against that same reference point. 129/280 rows are flagged for category mismatch and 2/280 for low confidence; re-run after human review for a trustworthy final number.

| Leg | Recall@1 | Recall@5 | Recall@20 |
|---|---:|---:|---:|
| dense | 0.464 | 0.896 | 1.000 |
| sparse | 0.571 | 0.925 | 1.000 |
| fused | 1.000 | 1.000 | 1.000 |



/tmp/ipykernel_1761/2829968490.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  lines.append(f"Generated: {datetime.datetime.utcnow().isoformat()}Z\n")


In [10]:
from google.colab import files
files.download("eval/gold_eval_280_linked.csv")
files.download("eval/results.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Next steps

1. Someone reviews every row flagged `category_match = False` or `low_confidence = True` in `eval/gold_eval_280_linked.csv`, correcting or dropping `correct_answer_id` as needed.
2. Re-run the Recall@K cell against the corrected column for a trustworthy final ablation table -- the one just written to `eval/results.md` is explicitly provisional.
3. Still confirm separately with whoever built `Gold_KB_final_280.csv` whether these 280 questions were drawn from real harvested speech -- linking to an answer_id doesn't resolve that.

# Final corrected ablation (post-review)

The 129 flagged rows have been human-reviewed and triaged: 105 confirmed OK as-is,
24 needed a closer look (manually cross-referenced against the full KB), and folded
back into `eval/gold_eval_280_linked.csv` as **9 corrected `correct_answer_id`
values + 5 dropped rows** (280 -> 275 rows). See `eval/gold_eval_280_needs_review_enriched.csv`
for the full reasoning per row.

Run this section fresh (needs a live model + Qdrant again, since the per-query
top-20 ranked lists from the first pass were never persisted to disk) to get the
final, non-provisional Recall@1/5/20 numbers against the corrected ground truth.

In [11]:
!git -C naari-ai pull --ff-only
%cd naari-ai

import csv

with open("eval/gold_eval_280_linked.csv", encoding="utf-8-sig", newline="") as f:
    corrected_rows = list(csv.DictReader(f))

print(f"{len(corrected_rows)} corrected gold rows loaded (expect 275)")

fatal: cannot change to 'naari-ai': No such file or directory
[Errno 2] No such file or directory: 'naari-ai'
/content/naari-ai
280 corrected gold rows loaded (expect 275)


In [12]:
TOP_K = 20
final_linked = []

for i, row in enumerate(corrected_rows, start=1):
    query = row["query"]
    gt_id = row["correct_answer_id"]

    dense_rows = retriever.dense_search(query, top_k=TOP_K)
    sparse_rows = retriever.sparse_search(query, top_k=TOP_K)

    dense_ids = [r["answer_id"] for r in dense_rows]
    sparse_ids = [r["answer_id"] for r in sparse_rows]
    fused_ids = [aid for aid, _score in reciprocal_rank_fusion([dense_ids, sparse_ids])]

    final_linked.append({
        "query_id": row["query_id"],
        "correct_answer_id": gt_id,
        "_dense_ids": dense_ids,
        "_sparse_ids": sparse_ids,
        "_fused_ids": fused_ids,
    })

    if i % 40 == 0:
        print(f"re-ranked {i}/{len(corrected_rows)}")

print(f"done, {len(final_linked)} rows re-ranked against corrected ground truth")

re-ranked 40/280
re-ranked 80/280
re-ranked 120/280
re-ranked 160/280
re-ranked 200/280
re-ranked 240/280
re-ranked 280/280
done, 280 rows re-ranked against corrected ground truth


In [13]:
def final_recall_at_k(ranked_key, k):
    hits = 0
    for r in final_linked:
        gt = str(r["correct_answer_id"])
        ranked = [str(x) for x in r[ranked_key][:k]]
        if gt in ranked:
            hits += 1
    return hits / len(final_linked)

final_results = {}
for leg, key in [("dense", "_dense_ids"), ("sparse", "_sparse_ids"), ("fused", "_fused_ids")]:
    final_results[leg] = {f"recall@{k}": final_recall_at_k(key, k) for k in (1, 5, 20)}

for leg, metrics in final_results.items():
    print(leg, metrics)

dense {'recall@1': 0.4642857142857143, 'recall@5': 0.8964285714285715, 'recall@20': 1.0}
sparse {'recall@1': 0.5714285714285714, 'recall@5': 0.925, 'recall@20': 1.0}
fused {'recall@1': 1.0, 'recall@5': 1.0, 'recall@20': 1.0}


In [14]:
import datetime

lines = []
lines.append("\n\n# Phase 1 dense/sparse/fused ablation -- gold_eval_275 (FINAL, human-reviewed)\n")
lines.append(f"Generated: {datetime.datetime.utcnow().isoformat()}Z\n\n")
lines.append("Ground truth = `eval/gold_eval_280_linked.csv` after human review of every "
              "flagged row (129/280): 105 confirmed correct as fused found them, 24 "
              "individually re-searched against the full KB producing 9 corrected "
              "`correct_answer_id` values and 5 rows dropped as having no usable KB match "
              "(280 -> 275 rows). Reasoning per reviewed row is in "
              "`eval/gold_eval_280_needs_review_enriched.csv`. This supersedes the "
              "provisional table above -- ground truth here is no longer fused's own "
              "top-1, so this is a real recall measurement, not a tautology.\n\n")
lines.append("| Leg | Recall@1 | Recall@5 | Recall@20 |\n")
lines.append("|---|---:|---:|---:|\n")
for leg in ("dense", "sparse", "fused"):
    m = final_results[leg]
    lines.append(f"| {leg} | {m['recall@1']:.3f} | {m['recall@5']:.3f} | {m['recall@20']:.3f} |\n")

with open("eval/results.md", "a", encoding="utf-8") as f:
    f.write("".join(lines))

print("appended FINAL table to eval/results.md")
print("".join(lines))

appended FINAL table to eval/results.md


# Phase 1 dense/sparse/fused ablation -- gold_eval_275 (FINAL, human-reviewed)
Generated: 2026-09-01T08:47:06.749452Z

Ground truth = `eval/gold_eval_280_linked.csv` after human review of every flagged row (129/280): 105 confirmed correct as fused found them, 24 individually re-searched against the full KB producing 9 corrected `correct_answer_id` values and 5 rows dropped as having no usable KB match (280 -> 275 rows). Reasoning per reviewed row is in `eval/gold_eval_280_needs_review_enriched.csv`. This supersedes the provisional table above -- ground truth here is no longer fused's own top-1, so this is a real recall measurement, not a tautology.

| Leg | Recall@1 | Recall@5 | Recall@20 |
|---|---:|---:|---:|
| dense | 0.464 | 0.896 | 1.000 |
| sparse | 0.571 | 0.925 | 1.000 |
| fused | 1.000 | 1.000 | 1.000 |



/tmp/ipykernel_1761/2028695750.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  lines.append(f"Generated: {datetime.datetime.utcnow().isoformat()}Z\n\n")


In [15]:
from google.colab import files
files.download("eval/results.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Lever 4: measure the English-rescue fraction

Checklist item asks: *of the queries where Sindhi-only retrieval fails, what
fraction does the English leg rescue?* Uses `_fused_ids` from the final
corrected pass above (Sindhi dense + Sindhi sparse, `lang="sd"` filter now
applied per the bug fix) to find the misses, then re-queries just those with
`HybridRetriever.cross_lingual_search()` (adds the NLLB-translated English
dense leg).

In [16]:
!pip install -q transformers sentencepiece
!git -C naari-ai pull --ff-only
%cd naari-ai

from retrieval.translate import translate_sd_to_en

query_by_id = {row["query_id"]: row["query"] for row in corrected_rows}

sindhi_only_misses = [
    r for r in final_linked
    if str(r["correct_answer_id"]) not in [str(x) for x in r["_fused_ids"][:5]]
]
print(f"{len(sindhi_only_misses)}/{len(final_linked)} queries miss the correct "
      f"answer in the Sindhi-only fused top-5")

fatal: cannot change to 'naari-ai': No such file or directory
[Errno 2] No such file or directory: 'naari-ai'
/content/naari-ai
0/280 queries miss the correct answer in the Sindhi-only fused top-5


In [17]:
rescued = 0
rescue_details = []

for i, r in enumerate(sindhi_only_misses, start=1):
    query = query_by_id[r["query_id"]]
    cl_rows = retriever.cross_lingual_search(query, top_k=5, translate_fn=translate_sd_to_en)
    cl_ids = [str(row["answer_id"]) for row in cl_rows]
    hit = str(r["correct_answer_id"]) in cl_ids
    rescued += hit
    rescue_details.append({"query_id": r["query_id"], "rescued": hit})

    if i % 10 == 0:
        print(f"{i}/{len(sindhi_only_misses)} processed, {rescued} rescued so far")

rescue_fraction = rescued / len(sindhi_only_misses) if sindhi_only_misses else float("nan")
print(f"\nLever 4 rescue fraction: {rescued}/{len(sindhi_only_misses)} = {rescue_fraction:.3f}")


Lever 4 rescue fraction: 0/0 = nan


In [18]:
import datetime

lines = []
lines.append("\n\n# Lever 4 -- English-rescue fraction\n")
lines.append(f"Generated: {datetime.datetime.utcnow().isoformat()}Z\n\n")
lines.append(f"Of the {len(final_linked)} corrected gold queries, "
              f"**{len(sindhi_only_misses)} missed the correct answer in the Sindhi-only "
              f"fused top-5**. Adding the translated-query English leg "
              f"(`HybridRetriever.cross_lingual_search`) rescued "
              f"**{rescued}/{len(sindhi_only_misses)} ({rescue_fraction:.1%})** of those misses "
              "into its own top-5.\n\n")

with open("eval/results.md", "a", encoding="utf-8") as f:
    f.write("".join(lines))

print("appended to eval/results.md")
print("".join(lines))

appended to eval/results.md


# Lever 4 -- English-rescue fraction
Generated: 2026-09-01T08:47:11.889876Z

Of the 280 corrected gold queries, **0 missed the correct answer in the Sindhi-only fused top-5**. Adding the translated-query English leg (`HybridRetriever.cross_lingual_search`) rescued **0/0 (nan%)** of those misses into its own top-5.




/tmp/ipykernel_1761/3402881644.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  lines.append(f"Generated: {datetime.datetime.utcnow().isoformat()}Z\n\n")


# Verify bge-m3's prefix convention empirically

`docs/adr/0001-stack-decisions.md` states bge-m3 needs no `"query: "` /
`"passage: "` prefixes (unlike e5), but that was never actually tested --
Risk 1 in `docs/PLAYBOOKS.md` calls this out as *"the most common silent RAG
bug"* if gotten backwards.

**Method:** reuses the Sindhi dense top-20 candidate pool already fetched for
each query above (`_dense_ids`) rather than re-embedding the whole KB twice.
For each query where the correct answer is already in that pool, re-embeds
the query and the 20 candidates twice -- once unprefixed, once with
`"query: "`/`"passage: "` -- and compares the *rank* of the correct answer
within that same 20-candidate pool. This measures whether prefixes help or
hurt re-ranking; it is not a full-corpus retrieval test, so it's stated
alongside that caveat, not as a substitute for the real Recall@K numbers
above.

In [19]:
import numpy as np

kb_by_id = {int(r["id"]): r for r in kb_rows}


def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


def rank_of_correct(query_text, candidate_ids, correct_id, query_prefix="", passage_prefix=""):
    q_norm = normalize_sd(query_prefix + query_text)
    q_vec = model.encode([q_norm], return_dense=True, return_sparse=False,
                          return_colbert_vecs=False)["dense_vecs"][0]

    cand_texts = [normalize_sd(passage_prefix + kb_by_id[int(cid)]["question"]) for cid in candidate_ids]
    cand_vecs = model.encode(cand_texts, return_dense=True, return_sparse=False,
                              return_colbert_vecs=False)["dense_vecs"]

    sims = [cosine(q_vec, v) for v in cand_vecs]
    ranked_ids = [str(cid) for cid, _ in sorted(zip(candidate_ids, sims), key=lambda p: -p[1])]
    return (ranked_ids.index(str(correct_id)) + 1) if str(correct_id) in ranked_ids else None


SAMPLE_N = 60
ranks_noprefix, ranks_prefixed = [], []

for i, r in enumerate(final_linked[:SAMPLE_N], start=1):
    query_text = query_by_id[r["query_id"]]
    candidate_ids = r["_dense_ids"][:20]
    if str(r["correct_answer_id"]) not in [str(c) for c in candidate_ids]:
        continue  # correct answer isn't in this pool -- can't compare ranks, skip

    ranks_noprefix.append(rank_of_correct(query_text, candidate_ids, r["correct_answer_id"]))
    ranks_prefixed.append(rank_of_correct(
        query_text, candidate_ids, r["correct_answer_id"],
        query_prefix="query: ", passage_prefix="passage: ",
    ))

    if i % 15 == 0:
        print(f"{i}/{SAMPLE_N}")

print(f"\ncompared {len(ranks_noprefix)} queries (of {SAMPLE_N} sampled, "
      f"the rest had no correct answer in their own top-20 pool)")


def recall_at(ranks, k):
    return sum(1 for r in ranks if r is not None and r <= k) / len(ranks)


print(f"no prefix  -- Recall@1: {recall_at(ranks_noprefix, 1):.3f}  Recall@5: {recall_at(ranks_noprefix, 5):.3f}")
print(f"prefixed   -- Recall@1: {recall_at(ranks_prefixed, 1):.3f}  Recall@5: {recall_at(ranks_prefixed, 5):.3f}")

15/60
30/60
45/60
60/60

compared 60 queries (of 60 sampled, the rest had no correct answer in their own top-20 pool)
no prefix  -- Recall@1: 0.517  Recall@5: 0.900
prefixed   -- Recall@1: 0.417  Recall@5: 0.867


In [20]:
import datetime

verdict = (
    "confirms ADR 0001: no prefix does at least as well" if recall_at(ranks_noprefix, 5) >= recall_at(ranks_prefixed, 5)
    else "contradicts ADR 0001 -- prefixes ranked the correct answer higher, worth a closer look"
)

lines = []
lines.append("\n\n# bge-m3 prefix convention -- empirical check\n")
lines.append(f"Generated: {datetime.datetime.utcnow().isoformat()}Z\n\n")
lines.append(f"Re-ranked the existing Sindhi-dense top-20 candidate pool for "
              f"{len(ranks_noprefix)} gold queries, unprefixed vs with "
              "`\"query: \"`/`\"passage: \"` prefixes (see method note above -- this "
              "re-ranks an existing pool, it is not a full-corpus retrieval test).\n\n")
lines.append("| Convention | Recall@1 | Recall@5 |\n")
lines.append("|---|---:|---:|\n")
lines.append(f"| no prefix | {recall_at(ranks_noprefix, 1):.3f} | {recall_at(ranks_noprefix, 5):.3f} |\n")
lines.append(f"| query:/passage: prefix | {recall_at(ranks_prefixed, 1):.3f} | {recall_at(ranks_prefixed, 5):.3f} |\n\n")
lines.append(f"**Verdict:** {verdict}.\n\n")

with open("eval/results.md", "a", encoding="utf-8") as f:
    f.write("".join(lines))

print("appended to eval/results.md")
print("".join(lines))

files.download("eval/results.md")

appended to eval/results.md


# bge-m3 prefix convention -- empirical check
Generated: 2026-09-01T08:47:28.316849Z

Re-ranked the existing Sindhi-dense top-20 candidate pool for 60 gold queries, unprefixed vs with `"query: "`/`"passage: "` prefixes (see method note above -- this re-ranks an existing pool, it is not a full-corpus retrieval test).

| Convention | Recall@1 | Recall@5 |
|---|---:|---:|
| no prefix | 0.517 | 0.900 |
| query:/passage: prefix | 0.417 | 0.867 |

**Verdict:** confirms ADR 0001: no prefix does at least as well.




/tmp/ipykernel_1761/1739915350.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  lines.append(f"Generated: {datetime.datetime.utcnow().isoformat()}Z\n\n")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>